# Stage 0 — PDF 재래스터화 (고해상도 PNG 생성)

`data/raw_pdf/*.pdf` (50장) → `data/drawings_pages/<stem>.png` 로 변환합니다.

**왜 다시 래스터화하나?** 기존 `data/drawings/images` 의 PNG(2526×1785)는 레포 밖 외부 단계로 만들어졌고,
작은 치수·공차·GD&T·거칠기 글자에는 해상도가 부족합니다. element 크롭이 Donut 에서 읽히려면
**크롭 1개당 충분한 픽셀**이 필요하므로 **300 DPI 이상**으로 다시 만듭니다.

- 커널: **kardi_env** (또는 PyMuPDF 가 설치된 아무 env)
- 엔진: PyMuPDF(`fitz`) 우선, 없으면 `pdf2image`(poppler 필요)

In [ ]:
# ── 의존성 확인/설치 ──────────────────────────────────────────────
# PyMuPDF(fitz) 우선. 미설치 시 아래 주석을 해제해 설치하세요.
import importlib, sys
try:
    import fitz  # PyMuPDF
    ENGINE = "pymupdf"
    print("PyMuPDF", fitz.__doc__.split("\n")[0])
except ImportError:
    try:
        import pdf2image  # noqa
        ENGINE = "pdf2image"
        print("pdf2image 사용 (poppler 필요)")
    except ImportError:
        ENGINE = None
        print("PDF 엔진 없음 → 설치 필요:")
        print("  pip install --no-deps pymupdf      # 권장")
        print("  # 또는: pip install pdf2image  (+ apt-get install poppler-utils)")

In [ ]:
# ── 설정 ─────────────────────────────────────────────────────────
from pathlib import Path
RAW_PDF_DIR = Path("../../data/raw_pdf")     # detection/ 에서 실행 기준
OUT_DIR     = Path("../../data/drawings_pages")
DPI         = 300                          # 300 이상 권장 (작은 글자 가독성)
ONLY_FIRST_PAGE = True                     # 도면은 보통 1페이지. 다중 페이지면 False
OUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(RAW_PDF_DIR.glob("*.pdf"))
print(f"PDF {len(pdfs)}개 발견 → {OUT_DIR}")

In [ ]:
# ── 래스터화 실행 ─────────────────────────────────────────────────
def rasterize_pymupdf(pdf_path, out_dir, dpi, only_first):
    import fitz
    doc = fitz.open(pdf_path)
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    saved = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(matrix=mat, alpha=False)
        stem = pdf_path.stem if (only_first or doc.page_count == 1) else f"{pdf_path.stem}_p{i+1:02d}"
        out = out_dir / f"{stem}.png"
        pix.save(out)
        saved.append(out)
        if only_first:
            break
    doc.close()
    return saved

def rasterize_pdf2image(pdf_path, out_dir, dpi, only_first):
    from pdf2image import convert_from_path
    pages = convert_from_path(str(pdf_path), dpi=dpi,
                              last_page=1 if only_first else None)
    saved = []
    for i, img in enumerate(pages):
        stem = pdf_path.stem if (only_first or len(pages) == 1) else f"{pdf_path.stem}_p{i+1:02d}"
        out = out_dir / f"{stem}.png"
        img.save(out)
        saved.append(out)
    return saved

assert ENGINE, "PDF 엔진을 먼저 설치하세요 (위 셀 참고)"
runner = rasterize_pymupdf if ENGINE == "pymupdf" else rasterize_pdf2image

total = 0
for pdf in pdfs:
    outs = runner(pdf, OUT_DIR, DPI, ONLY_FIRST_PAGE)
    total += len(outs)
print(f"완료: {total}개 PNG → {OUT_DIR}")

In [ ]:
# ── 검증: 개수 + 샘플 1장 + 해상도 ───────────────────────────────
from PIL import Image
import matplotlib.pyplot as plt
pngs = sorted(OUT_DIR.glob("*.png"))
print(f"생성된 PNG: {len(pngs)}개")
if pngs:
    im = Image.open(pngs[0])
    print(f"예시 {pngs[0].name}  크기={im.size}  (가로 3000px+ 권장)")
    plt.figure(figsize=(14, 10)); plt.imshow(im); plt.axis("off")
    plt.title(pngs[0].name); plt.show()